In [10]:
import os
from io import BytesIO
from elevenlabs.client import ElevenLabs
from elevenlabs import play
from dotenv import load_dotenv

load_dotenv()

ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY")
elevenlabs = ElevenLabs(
    api_key=ELEVENLABS_API_KEY
)

paragraphs = [
    "El Siglo de Oro español, que abarcó desde finales del siglo XV hasta mediados del XVII, ",
    "representa uno de los períodos más brillantes y extraordinarios de la cultura occidental.",
    "Durante esta época dorada, España no solo dominó los mares y conquistó vastos territorios, ",
    "sino que también alumbró una constelación de genios literarios que transformaron para siempre las letras universales.",
    "Cervantes inmortalizó la condición humana con su Don Quijote, ",
    "mientras que Lope de Vega y Calderón de la Barca revolucionaron el teatro europeo ",
    "con su creatividad desbordante y su profunda comprensión del alma española."
]


request_ids = []
audio_buffers = []

for paragraph in paragraphs:
    # Usually we get back a stream from the convert function, but with_raw_response is
    # used to get the headers from the response
    with elevenlabs.text_to_speech.with_raw_response.convert(
        text=paragraph,
        voice_id="sDuUJMeNJR828mXTRrDh",
        model_id="eleven_multilingual_v2",
        previous_request_ids=request_ids,
    ) as response:
        request_ids.append(response._response.headers.get("request-id"))

        # response._response.headers also contains useful information like 'character-cost',
        # which shows the cost of the generation in characters.

        audio_data = b''.join(chunk for chunk in response.data)

        audio_buffers.append(BytesIO(audio_data))
    
combined_stream = BytesIO(b''.join(buffer.getvalue() for buffer in audio_buffers))
play.play(combined_stream)


ApiError: headers: {'date': 'Sun, 14 Sep 2025 22:02:10 GMT', 'server': 'uvicorn', 'content-length': '136', 'content-type': 'application/json', 'access-control-allow-origin': '*', 'access-control-allow-headers': '*', 'access-control-allow-methods': 'POST, PATCH, OPTIONS, DELETE, GET, PUT', 'access-control-max-age': '600', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'x-trace-id': 'd7a54f93a5fee94aa7711f6544436d5c', 'x-region': 'us-central1', 'via': '1.1 google, 1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'}, status_code: 400, body: {'detail': {'status': 'too_many_history_item_ids', 'message': 'A maximum of 3 previous_history_item_ids can be send but received 4 items.'}}

try with max 3 slices

In [12]:
from collections import deque

request_ids = deque(maxlen=3)
audio_buffers = []

for paragraph in paragraphs:
    with elevenlabs.text_to_speech.with_raw_response.convert(
        text=paragraph,
        voice_id="sDuUJMeNJR828mXTRrDh",
        model_id="eleven_multilingual_v2",
        previous_request_ids=list(request_ids),  # send <=3
    ) as response:
        # prefer history-item-id, fallback to request-id
        rid = response._response.headers.get("history-item-id") or response._response.headers.get("request-id")
        if rid:
            request_ids.append(rid)

        audio_data = b"".join(chunk for chunk in response.data)
        audio_buffers.append(BytesIO(audio_data))

combined_stream = BytesIO(b''.join(buffer.getvalue() for buffer in audio_buffers))
play.play(combined_stream)


but e.g. turbo does work with previous config

In [14]:
for paragraph in paragraphs:
    # Usually we get back a stream from the convert function, but with_raw_response is
    # used to get the headers from the response
    with elevenlabs.text_to_speech.with_raw_response.convert(
        text=paragraph,
        voice_id="sDuUJMeNJR828mXTRrDh",
        model_id="eleven_turbo_v2_5",
        previous_request_ids=request_ids,
    ) as response:
        request_ids.append(response._response.headers.get("request-id"))

        # response._response.headers also contains useful information like 'character-cost',
        # which shows the cost of the generation in characters.

        audio_data = b''.join(chunk for chunk in response.data)

        audio_buffers.append(BytesIO(audio_data))
    
combined_stream = BytesIO(b''.join(buffer.getvalue() for buffer in audio_buffers))
play.play(combined_stream)